In [ ]:
######################
# @title MODEL EfficientNet-B0 ( Without Aug + Mixed Precision)
######################
import os
import json
import copy
import time
import math
import shutil
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights
from torch.amp import autocast, GradScaler
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


LOCAL_RUN_NAME = "model_i10_efficientNetB0_noaug_partial"


BASE_DIR = Path.cwd().parent.parent.parent
LOCAL_DATA_DIR = BASE_DIR / "data"

OUTPUT_DIR_BASE = BASE_DIR / "outputs" / "image_modeling" / LOCAL_RUN_NAME
MODEL_DIR_BASE = BASE_DIR / "data" / "models" / LOCAL_RUN_NAME
FIGURE_DIR_BASE = BASE_DIR / "figures" / LOCAL_RUN_NAME

OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)
MODEL_DIR_BASE.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_BASE.mkdir(parents=True, exist_ok=True)

DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
train_df = pd.read_csv(SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
scaler = GradScaler("cuda", enabled=use_amp)

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Mixed precision enabled:", use_amp)

# ------------------------------------------------------------
# Local save dirs for this run
# ------------------------------------------------------------
LOCAL_RUN_NAME = "efficientnet_b0_pretrained_no_224_aug_amp"
LOCAL_OUTPUT_ROOT = LOCAL_DATA_DIR / "outputs" / LOCAL_RUN_NAME
LOCAL_MODEL_ROOT = LOCAL_DATA_DIR / "models" / LOCAL_RUN_NAME

for d in [
    LOCAL_OUTPUT_ROOT, LOCAL_MODEL_ROOT,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Local output root:", LOCAL_OUTPUT_ROOT)

# ------------------------------------------------------------
# Transforms / loaders
# ------------------------------------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

train_transform_H = transforms.Compose([
    transforms.Resize((IMAGE_SIZE , IMAGE_SIZE )),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform_H = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class RakutenImageDataset(Dataset):
    def __init__(self, df, transform=None, path_col="image_path_local", label_col="label_id"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.path_col = path_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, self.path_col]
        label = int(self.df.loc[idx, self.label_col])
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

train_dataset_H = RakutenImageDataset(train_df, transform=train_transform_H)
val_dataset_H = RakutenImageDataset(val_df, transform=val_transform_H)

train_loader_H = DataLoader(
    train_dataset_H,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader_H = DataLoader(
    val_dataset_H,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
weights = EfficientNet_B0_Weights.DEFAULT
model_H = models.efficientnet_b0(weights=weights)

for param in model_H.parameters():
    param.requires_grad = False

for param in model_H.features[-1].parameters():
    param.requires_grad = True

in_features = model_H.classifier[1].in_features
model_H.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, len(label2id))
)

for param in model_H.classifier.parameters():
    param.requires_grad = True

model_H = model_H.to(device)
print(model_H)

trainable_params = sum(p.numel() for p in model_H.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model_H.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"Total params    : {all_params:,}")

# ------------------------------------------------------------
# Loss, optimizer, scheduler
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_H.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
# MODEL_NAME = "EfficientNet_B0_Pretrained_224_No_Aug_AMP"
MAX_EPOCHS = 18
EARLY_STOPPING_PATIENCE = 6

BEST_CHECKPOINT_LOCAL = LOCAL_MODEL_ROOT / "best_checkpoint.pt"

LAST_CHECKPOINT_LOCAL = LOCAL_MODEL_ROOT / "last_checkpoint.pt"

BEST_WEIGHTS_LOCAL = LOCAL_MODEL_ROOT / "best_model_state_dict.pt"

def save_full_checkpoint(checkpoint, local_path):
    torch.save(checkpoint, local_path)
    shutil.copy2(local_path)

def load_full_checkpoint(load_path, model, optimizer=None, scheduler=None, scaler=None, device="cpu"):
    checkpoint = torch.load(load_path, map_location=device, weights_only=False)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

        if optimizer is not None and "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        if scheduler is not None and "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        if scaler is not None and "scaler_state_dict" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])

        if "torch_rng_state" in checkpoint:
            torch.set_rng_state(checkpoint["torch_rng_state"])

        if "numpy_rng_state" in checkpoint:
            np.random.set_state(checkpoint["numpy_rng_state"])

        if "python_rng_state" in checkpoint:
            random.setstate(checkpoint["python_rng_state"])

        if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
            torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])

        return checkpoint
    else:
        model.load_state_dict(checkpoint)
        return {
            "epoch": 0,
            "best_epoch": 0,
            "best_macro_f1": -np.inf,
            "history": [],
            "epochs_without_improvement": 0
        }

def plot_history(history_df, loss_save_local):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{LOCAL_RUN_NAME} - Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(loss_save_local, dpi=200, bbox_inches="tight")
    shutil.copy2(loss_save_local)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{LOCAL_RUN_NAME} - Accuracy")
    plt.legend()
    plt.tight_layout()
    acc_local = loss_save_local.parent / "training_accuracy.png"
    plt.savefig(acc_local, dpi=200, bbox_inches="tight")
    shutil.copy2(acc_local)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="Train Macro F1")
    plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="Val Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(f"{LOCAL_RUN_NAME} - Macro F1")
    plt.legend()
    plt.tight_layout()
    f1_local = loss_save_local.parent / "training_macro_f1.png"
    plt.savefig(f1_local, dpi=200, bbox_inches="tight")
    shutil.copy2(f1_local)
    plt.show()

def run_epoch(model, loader, criterion, optimizer=None, phase="train"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds = []
    all_true = []

    for batch_idx, (images, labels) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with autocast("cuda", enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

        if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == len(loader):
            print(f"{phase} batch {batch_idx}/{len(loader)} done")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_macro_f1 = f1_score(all_true, all_preds, average="macro")
    epoch_weighted_f1 = f1_score(all_true, all_preds, average="weighted")

    return epoch_loss, epoch_acc, epoch_macro_f1, epoch_weighted_f1, np.array(all_true), np.array(all_preds)

def predict_with_proba(model, loader):
    model.eval()
    all_probs = []
    all_preds = []
    all_true = []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(loader, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(images)
                probs = torch.softmax(logits, dim=1)

            preds = torch.argmax(probs, dim=1)

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

            if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == len(loader):
                print(f"predict batch {batch_idx}/{len(loader)} done")

    y_proba = np.concatenate(all_probs)
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    return y_true, y_pred, y_proba

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
history = []
best_macro_f1 = -np.inf
best_epoch = -1
epochs_without_improvement = 0
start_epoch = 1

resume_path = None

if LAST_CHECKPOINT_LOCAL.exists():
    resume_path = LAST_CHECKPOINT_LOCAL
elif BEST_CHECKPOINT_LOCAL.exists():
    resume_path = BEST_CHECKPOINT_LOCAL

if resume_path is not None:
    print(f"Existing checkpoint found: {resume_path}")
    loaded_ckpt = load_full_checkpoint(
        resume_path,
        model_H,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
    )
    start_epoch = int(loaded_ckpt.get("epoch", 0)) + 1
    history = loaded_ckpt.get("history", [])
    best_macro_f1 = float(loaded_ckpt.get("best_macro_f1", -np.inf))
    best_epoch = int(loaded_ckpt.get("best_epoch", -1))
    epochs_without_improvement = int(loaded_ckpt.get("epochs_without_improvement", 0))
    print(f"Resuming from epoch {start_epoch}")
    print(f"Best epoch so far: {best_epoch}")
    print(f"Best val macro F1 so far: {best_macro_f1:.6f}")
else:
    print("No checkpoint found. Starting from scratch.")

start_time = time.time()

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    train_loss, train_acc, train_macro_f1, train_weighted_f1, _, _ = run_epoch(
        model_H, train_loader_H, criterion, optimizer=optimizer, phase="train"
    )

    val_loss, val_acc, val_macro_f1, val_weighted_f1, _, _ = run_epoch(
        model_H, val_loader_H, criterion, optimizer=None, phase="val"
    )

    scheduler.step(val_macro_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_macro_f1": train_macro_f1,
        "train_weighted_f1": train_weighted_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_macro_f1,
        "val_weighted_f1": val_weighted_f1,
        "lr": current_lr
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | "
        f"train_macro_f1={train_macro_f1:.4f} | val_macro_f1={val_macro_f1:.4f} | "
        f"lr={current_lr:.6f}"
    )

    improved = val_macro_f1 > best_macro_f1

    if improved:
        best_macro_f1 = val_macro_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        best_checkpoint = {
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_macro_f1": float(best_macro_f1),
            "epochs_without_improvement": int(epochs_without_improvement),
            "model_state_dict": copy.deepcopy(model_H.state_dict()),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
            "model_name": LOCAL_RUN_NAME,
            "torch_rng_state": torch.get_rng_state(),
            "numpy_rng_state": np.random.get_state(),
            "python_rng_state": random.getstate()
        }

        if torch.cuda.is_available():
            best_checkpoint["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

        save_full_checkpoint(best_checkpoint, BEST_CHECKPOINT_LOCAL)
        torch.save(model_H.state_dict(), BEST_WEIGHTS_LOCAL)
        shutil.copy2(BEST_WEIGHTS_LOCAL)

        print(f"  -> New best model saved at epoch {epoch} (val_macro_f1={val_macro_f1:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")

    last_checkpoint = {
        "epoch": epoch,
        "best_epoch": best_epoch,
        "best_macro_f1": float(best_macro_f1),
        "epochs_without_improvement": int(epochs_without_improvement),
        "model_state_dict": model_H.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "model_name": LOCAL_RUN_NAME,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate()
    }

    if torch.cuda.is_available():
        last_checkpoint["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    save_full_checkpoint(last_checkpoint, LAST_CHECKPOINT_LOCAL)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered after epoch {epoch}")
        break

total_time = time.time() - start_time
print(f"\nTraining finished in {total_time/60:.2f} minutes")
print(f"Best epoch: {best_epoch}")
print(f"Best val macro F1: {best_macro_f1:.6f}")


if BEST_CHECKPOINT_LOCAL.exists():
    best_loaded = torch.load(BEST_CHECKPOINT_LOCAL, map_location=device, weights_only=False)
elif BEST_CHECKPOINT_LOCAL.exists():
    best_loaded = torch.load(BEST_CHECKPOINT_LOCAL, map_location=device, weights_only=False)
else:
    best_loaded = torch.load(LAST_CHECKPOINT_LOCAL, map_location=device, weights_only=False)

if "model_state_dict" in best_loaded:
    model_H.load_state_dict(best_loaded["model_state_dict"])
else:
    model_H.load_state_dict(best_loaded)

y_true, y_pred, y_proba = predict_with_proba(model_H, val_loader_H)

# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------
final_accuracy = accuracy_score(y_true, y_pred)
final_macro_f1 = f1_score(y_true, y_pred, average="macro")
final_weighted_f1 = f1_score(y_true, y_pred, average="weighted")

metrics_df = pd.DataFrame([{
    "model_name": LOCAL_RUN_NAME,
    "image_size": IMAGE_SIZE,
    "augmentation": True,
    "architecture": "EfficientNet_B0_Pretrained",
    "epochs_trained": len(history),
    "best_epoch": best_epoch,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "initial_lr": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "accuracy": final_accuracy,
    "macro_f1": final_macro_f1,
    "weighted_f1": final_weighted_f1,
    "training_minutes_this_session": total_time / 60,
    "mixed_precision": use_amp
}])

print("\nFinal validation metrics:")
print(metrics_df)

# ------------------------------------------------------------
# Save metrics/history
# ------------------------------------------------------------
history_df = pd.DataFrame(history)

metrics_local = LOCAL_OUTPUT_ROOT / "metrics.csv"
history_local = LOCAL_OUTPUT_ROOT / "training_history.csv"

metrics_df.to_csv(metrics_local, index=False)
history_df.to_csv(history_local, index=False)
shutil.copy2(metrics_local)
shutil.copy2(history_local)

# ------------------------------------------------------------
# Save predictions/probabilities
# ------------------------------------------------------------
pred_df = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred
})

pred_local = LOCAL_OUTPUT_ROOT / "val_predictions.csv"
pred_df.to_csv(pred_local, index=False)
shutil.copy2(pred_local)

proba_local = LOCAL_OUTPUT_ROOT / "y_proba.npy"
np.save(proba_local, y_proba)
shutil.copy2(proba_local)

# ------------------------------------------------------------
# Save classification report
# ------------------------------------------------------------
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().reset_index().rename(columns={"index": "class_or_avg"})

report_local = LOCAL_OUTPUT_ROOT / "classification_report.csv"
report_df.to_csv(report_local, index=False)
shutil.copy2(report_local)

# ------------------------------------------------------------
# Save confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title(f"{LOCAL_RUN_NAME} - Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()

cm_local = LOCAL_FIG_ROOT / "confusion_matrix.png"
plt.savefig(cm_local, dpi=200, bbox_inches="tight")
shutil.copy2(cm_local)
plt.show()

# ------------------------------------------------------------
# Save training curves
# ------------------------------------------------------------
loss_local = LOCAL_FIG_ROOT / "training_loss.png"
plot_history(history_df, loss_local)

# ------------------------------------------------------------
# Save run metadata
# ------------------------------------------------------------
metadata = {
    "model_name": LOCAL_RUN_NAME,
    "image_size": IMAGE_SIZE,
    "augmentation": True,
    "architecture": "EfficientNet_B0_Pretrained",
    "epochs_trained": int(len(history)),
    "best_epoch": int(best_epoch),
    "batch_size": int(BATCH_SIZE),
    "optimizer": "AdamW",
    "initial_lr": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
    "accuracy": float(final_accuracy),
    "macro_f1": float(final_macro_f1),
    "weighted_f1": float(final_weighted_f1),
    "training_minutes_this_session": float(total_time / 60),
    "resumed_from_checkpoint": resume_path is not None,
    "resume_path": str(resume_path) if resume_path is not None else None,
    "best_checkpoint_local": str(BEST_CHECKPOINT_LOCAL),
    "last_checkpoint_local": str(LAST_CHECKPOINT_LOCAL),
    "trainable_parameters": int(trainable_params),
    "total_parameters": int(all_params),
    "mixed_precision": bool(use_amp)
}

meta_local = LOCAL_OUTPUT_ROOT / "run_metadata.json"
with open(meta_local, "w") as f:
    json.dump(metadata, f, indent=2)
shutil.copy2(meta_local)

print("\nSaved files:")
print("Best model local     :", BEST_CHECKPOINT_LOCAL)
print("Best weights local   :", BEST_WEIGHTS_LOCAL)
print("Last checkpoint local:", LAST_CHECKPOINT_LOCAL)
print("Metrics local        :", metrics_local)
print("History local        :", history_local)
print("Predictions          :", pred_local)
print("Probabilities        :", proba_local)
print("Class report         :", report_local)
print("Conf matrix fig      :", cm_local)
print("Metadata             :", meta_local)